In [1]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler, BitArray
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays


from ansatzmap import get_zigzag_physical_layout

from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps


In [2]:
BasisDirs=glob('data/*')

In [3]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [4]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [5]:
BasisSets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

In [6]:
# os.mkdir('counts')
service = QiskitRuntimeService()

management.get:WARNING:2026-04-01 10:11:14,726: Loading default saved account


In [7]:
"False"

'False'

In [8]:
def run(pathxyz,name,basis,n_electrons,num_orbitals,L,k):
    filecontents=f"""
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays
from qiskit.primitives import StatevectorSampler, BitArray


from ansatzmap import get_zigzag_physical_layout

from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps    

ampdict = GrabAmps("{name}","{basis}")

t1, t2 = ampdict["{k}"]
initDDLUCJ = DDLUCJ(StructurePath="{pathxyz}", 
                    BasisSet="{basis}", 
                    NElec=int({n_electrons}),
                    NOrb=int({num_orbitals}),
                    injected=True,
                    t1=t1, 
                    t2=t2,
                    n_reps = int({L}),
                    optimization_level=3,
                    temp_dir="./",
                    clean_temp_dir=True,
                    n_jobs=64,
                    num_batches = 10,
                    max_iterations=5,
                    samples_per_batch=1000,
                    verbose=False)

counts = np.load(f"../counts/{name}_LUCJ_L{L}_{basis}_{k}.npz")
bitstrings = counts['bitstrings']
probarr = counts['probarr']
bitstrings = BitArray.from_bool_array(bitstrings)

result_history, result = initDDLUCJ(postprocess=True,BitArray=bitstrings) 

new_energy = result.energy + initDDLUCJ.nuclear_repulsion_energy
EnergyPath = f"../energies/{name}_LUCJ_L{L}_{basis}_{k}.txt" 
with open(EnergyPath,'w') as f:
    new_row={{"Basis Set": "{basis}", "Molecule": "{name}", "Method": f"LUCJ(L={L})/{k}", "Energy": new_energy}}
    for k,v in new_row.items():
        f.write(f"{{v}}\\n")

"""
    with open(f"./postprocess/{name}_LUCJ_L{L}_{basis}_{k}.py",'w') as f:
        f.write(filecontents)

    runfile=f"""#!/bin/bash
#SBATCH --time=0-8:00:00
#SBATCH -J {name}_LUCJ_L{L}_{basis}_{k}
#SBATCH --account=rrg-jacobsen-ab
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=64
#SBATCH --mem-per-cpu=1000M
#SBATCH --error=job.e%J
#SBATCH --output=job.o%j



echo 'About to run python file'
module load python/3.10
module load openmpi
module load symengine rust
module load hdf5
module load openblas
source /lustre09/project/6004825/gjones/ENV/bin/activate
export LD_LIBRARY_PATH=$EBROOTOPENBLAS/lib:$LD_LIBRARY_PATH

export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK
export OPENBLAS_NUM_THREADS=$SLURM_CPUS_PER_TASK
export MKL_NUM_THREADS=$SLURM_CPUS_PER_TASK
export NUMEXPR_NUM_THREADS=$SLURM_CPUS_PER_TASK
echo "Running in directory: $(pwd)"

python {name}_LUCJ_L{L}_{basis}_{k}.py 
echo "File run"    
"""
    with open(f"./postprocess/{name}_LUCJ_L{L}_{basis}_{k}.sh",'w') as f:
            f.write(runfile)


In [9]:
# os.mkdir('energies')

In [10]:
postprocessed = []
for i in tqdm(sorted(glob("./jobids/*txt")),desc='Running'):
    i.split("/")[-1].replace('.txt','').split('_')
    with open(i,'r') as f:
        name,basis,k,L,JobID = [i.strip() for i in f.readlines()]
    print(name,basis,k,L,JobID)
    moldict = moldf[moldf['molecule']==name]

    n_electrons=moldict['n_electrons'].values[0]
    num_orbitals=moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    pathxyz = os.path.join("../../../../classical/structures/",xyzname)



    JobPath = f"../jobids/{name}_LUCJ_L{L}_{basis}_{k}.txt" 
    EnergyPath = f"../energies/{name}_LUCJ_L{L}_{basis}_{k}.txt" 

    # if os.path.exists(JobPath)==True and os.path.exists(EnergyPath)==False:
    print(f"Running {name}_LUCJ_L{L}_{basis}_{k}")
    run(pathxyz,name,basis,n_electrons,num_orbitals,L,k)

Running:  61%|███████████████████████████████████████████████████▎                                | 660/1080 [00:00<00:00, 3305.41it/s]

(Z)-1-fluoroprop-1-ene STO-3G CCSD 1 d3lvg6j4kkus739d5sug
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_CCSD
(Z)-1-fluoroprop-1-ene STO-3G ML 1 d3lvi0gdd19c7397ei4g
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_ML
(Z)-1-fluoroprop-1-ene STO-3G ML_exact 1 d3lvjr1fk6qs73e7cvq0
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_ML_exact
(Z)-1-fluoroprop-1-ene STO-3G MP2 1 d3lvec0dd19c7397eeqg
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_MP2
(Z)-1-fluoroprop-1-ene STO-3G random 1 d3lvm6r4kkus739d62f0
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_random
(Z)-1-fluoroprop-1-ene STO-3G zeroes 1 d3lvldpfk6qs73e7d19g
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_zeroes
(Z)-1-fluoroprop-1-ene aug-cc-pVDZ CCSD 1 d3m0268dd19c7397f14g
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_aug-cc-pVDZ_CCSD
(Z)-1-fluoroprop-1-ene aug-cc-pVDZ ML 1 d3m02t83qtks738cr780
Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_aug-cc-pVDZ_ML
(Z)-1-fluoroprop-1-ene aug-cc-pVDZ ML_exact 1 d3m03jo3qtks738cr7sg
Running (Z)-1-fluoroprop-1-ene_LUCJ

Running: 100%|███████████████████████████████████████████████████████████████████████████████████| 1080/1080 [00:00<00:00, 3383.11it/s]

formaldehyde cc-pVDZ MP2 2 d3lffqodd19c7396vkog
Running formaldehyde_LUCJ_L2_cc-pVDZ_MP2
formaldehyde cc-pVDZ random 2 d3lflhhfk6qs73e6u3kg
Running formaldehyde_LUCJ_L2_cc-pVDZ_random
formaldehyde cc-pVDZ zeroes 2 d3lfkm1fk6qs73e6u2n0
Running formaldehyde_LUCJ_L2_cc-pVDZ_zeroes
formaldehyde STO-3G CCSD 3 d3lf6b34kkus739cmf40
Running formaldehyde_LUCJ_L3_STO-3G_CCSD
formaldehyde STO-3G ML 3 d3lfa783qtks738cbgqg
Running formaldehyde_LUCJ_L3_STO-3G_ML
formaldehyde STO-3G ML_exact 3 d3lfcmj4kkus739cmlo0
Running formaldehyde_LUCJ_L3_STO-3G_ML_exact
formaldehyde STO-3G MP2 3 d3lf4or4kkus739cmdhg
Running formaldehyde_LUCJ_L3_STO-3G_MP2
formaldehyde STO-3G random 3 d3lferpfk6qs73e6tsv0
Running formaldehyde_LUCJ_L3_STO-3G_random
formaldehyde STO-3G zeroes 3 d3lfdr03qtks738cbkn0
Running formaldehyde_LUCJ_L3_STO-3G_zeroes
formaldehyde aug-cc-pVDZ CCSD 3 d3lfn7hfk6qs73e6u5b0
Running formaldehyde_LUCJ_L3_aug-cc-pVDZ_CCSD
formaldehyde aug-cc-pVDZ ML 3 d3lfnob4kkus739cn14g
Running formaldehyde_LUCJ_L